# 02 — The control: reproducing the published result

`run_conventional` implements the modal pipeline from notebook 01: a single unstratified
80/20 split, one-hot encoding, StandardScaler fit on all rows before the split, default
decision tree / logistic regression / random forest / SVM, with random forest as the
headline.

Choices the survey did not settle, stated here: every column is scaled after one-hot
encoding; `get_dummies` keeps all levels; model `random_state` is set to the split seed so
runs are reproducible (most surveyed notebooks leave it unset).

In [1]:
import json
import pandas as pd
from heart_audit.conventional import reproduction_gate
from heart_audit.data import PROJECT_ROOT
from heart_audit.plots import control_distribution

control = pd.read_csv(PROJECT_ROOT / "results" / "control_seeds.csv")   # scripts/reproduce_control.py
summary = json.loads((PROJECT_ROOT / "survey" / "modal_pipeline.json").read_text(encoding="utf-8"))
coding = pd.read_csv(PROJECT_ROOT / "survey" / "coding.csv", keep_default_na=False, na_values=[""])
control.drop(columns="seed").describe().round(4)

,decision_tree,logistic_regression,random_forest,svm
count,1000.0000,1000.0000,1000.0000,1000.0000
mean,0.7928,0.8610,0.8692,0.8647
std,0.0277,0.0230,0.0226,0.0224
min,0.7011,0.7880,0.7935,0.7880
25%,0.7717,0.8478,0.8533,0.8533
50%,0.7935,0.8587,0.8696,0.8641
75%,0.8098,0.8750,0.8859,0.8804
max,0.8804,0.9348,0.9348,0.9402


## The reproduction gate (pre-registered in the spec)

Over 1,000 split seeds, the median surveyed accuracy must lie inside the central 95% of the
control's random-forest accuracy distribution.

In [2]:
targets = {
    "primary": summary["primary"]["accuracy"]["median"],
    "literal E4": summary["sensitivity_literal_e4"]["accuracy"]["median"],
    "D2 best shown": summary["sensitivity_d2_best_shown_accuracy"]["median"],
}
rows = []
for name, t in targets.items():
    lo, hi, passed = reproduction_gate(control["random_forest"], t)
    rows.append({"reading": name, "survey median": t, "control 2.5%": lo, "control 97.5%": hi, "passed": passed})
gate = pd.DataFrame(rows)
gate

,reading,survey median,control 2.5%,control 97.5%,passed
0,primary,0.8610,0.820652,0.913043,True
1,literal E4,0.8632,0.820652,0.913043,True
2,D2 best shown,0.8658,0.820652,0.913043,True


In [3]:
lo, hi, _ = reproduction_gate(control["random_forest"], targets["primary"])
control_distribution(control["random_forest"], coding["accuracy_value"].dropna(), targets["primary"],
                     (lo, hi), n_test=184, path=PROJECT_ROOT / "images" / "control_seed_distribution.png")

WindowsPath('C:/Users/ethan/Desktop/Coding Projects/heart-disease-audit/images/control_seed_distribution.png')

![](../images/control_seed_distribution.png)

## What this shows

The control reproduces the published results: every surveyed median falls inside its central
95%. The same distribution also shows that a single-split accuracy is mostly a property of
the split. The pipeline and data stay fixed, and the random forest's accuracy still ranges
across the band above depending only on which 184 rows are held out. Notebook 04 (T4, T5)
measures what that does to model rankings and to reporting the best of several models.

In [4]:
print(f"random forest over {len(control)} seeds: median {control.random_forest.median():.4f}, "
      f"central 95% [{lo:.4f}, {hi:.4f}], width {hi - lo:.4f}")

random forest over 1000 seeds: median 0.8696, central 95% [0.8207, 0.9130], width 0.0924
